# Treino de um modelo XGBoost com métricas e matriz de confusão

Neste notebook vamos:

1. Carregar o arquivo de features (`features_old.csv`);
2. Preparar os dados (remover colunas que atrapalham e tratar duplicatas);
3. Separar os dados em treino e teste;
4. Treinar um **XGBoost** (classificador binário);
5. Avaliar com métricas e **matriz de confusão**;
6. Salvar o modelo em **`.joblib`**.

**Sobre os dados:** cada linha representa a imagem de um documento. As colunas indicam a posição de um rosto detectado (`x_face`, `y_face`, `w_face`, `h_face`), a quantidade de palavras lidas por OCR e se certas palavras apareceram (`estudantil`, `unicamp`, `CPF`, etc.). A coluna `label` é o que queremos prever (0 ou 1).

## 1. Importando as bibliotecas

In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

## 2. Configurações

Aqui ficam os parâmetros que você pode querer mudar. Assim, o resto do notebook não precisa ser alterado.

In [5]:
CAMINHO_CSV = "features_old.csv"        # caminho do arquivo de features
COLUNA_ALVO = "label"                   # coluna que queremos prever
COLUNAS_PARA_IGNORAR = ["nome_arquivo"] # colunas que NÃO devem entrar no modelo
TAMANHO_TESTE = 0.2                     # 20% dos dados ficam para o teste
SEMENTE = 42                            # garante resultados reproduzíveis
REMOVER_DUPLICADOS = True               # ver explicação na seção 4
CAMINHO_MODELO = "modelo_xgboost.joblib"

## 3. Carregando e conhecendo os dados

In [6]:
df = pd.read_csv(CAMINHO_CSV)
print("Formato (linhas, colunas):", df.shape)
df.head()

Formato (linhas, colunas): (1000, 20)


,nome_arquivo,x_face,y_face,w_face,h_face,quantidade_palavras,estudantil,validade,unicamp,universidade,estadual,matricula,documento,nacional,estudante,UNE,CPF,RG,data,label
0,aug_doc1_imagem_178.jpg,526.0,494.0,98.0,98.0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,1
1,aug_doc1_imagem_47.jpg,64.0,1085.0,27.0,27.0,15,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,aug_doc1_imagem_53.jpg,NaN,NaN,NaN,NaN,11,1,0,1,0,0,0,0,0,0,0,0,0,0,1
3,aug_doc1_imagem_190.jpg,325.0,554.0,51.0,51.0,14,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,aug_doc1_imagem_189.jpg,64.0,1085.0,27.0,27.0,14,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [7]:
# Distribuição das classes: queremos saber se os dados estão balanceados
print(df[COLUNA_ALVO].value_counts())
print()
print(df[COLUNA_ALVO].value_counts(normalize=True).round(3))

label
1    500
0    500
Name: count, dtype: int64

label
1    0.5
0    0.5
Name: proportion, dtype: float64


In [8]:
# Valores ausentes por coluna
ausentes = df.isna().sum()
ausentes[ausentes > 0]

x_face    280
y_face    280
w_face    280
h_face    280
dtype: int64

As colunas do rosto (`x_face`, `y_face`, `w_face`, `h_face`) têm valores ausentes. Isso acontece quando o detector **não encontrou nenhum rosto** na imagem. Isso não é um erro: é uma informação útil. O XGBoost sabe lidar com valores ausentes (`NaN`) por conta própria, então não precisamos preencher nada. Mesmo assim, vamos criar uma coluna extra, `rosto_detectado`, para deixar essa informação explícita.

## 4. Preparando os dados

Antes de treinar, três cuidados importantes.

**a) Remover `nome_arquivo` (vazamento de dados).** Os nomes dos arquivos seguem um padrão que depende da classe (por exemplo, `aug_doc1_...` só aparece em uma delas). Se o modelo enxergasse esse nome, ele "colaria" na prova em vez de aprender com as features reais. Isso se chama *data leakage* e geraria métricas falsamente perfeitas.

**b) Remover colunas constantes.** Colunas que valem sempre o mesmo número (por exemplo, sempre 0) não ajudam o modelo em nada.

**c) Tratar linhas duplicadas.** As imagens têm prefixo `aug`, o que indica *data augmentation* (variações da mesma imagem). Muitas variações geram exatamente as mesmas features. Se uma linha cair no treino e a cópia idêntica cair no teste, o modelo "já viu" a resposta e a métrica fica inflada. Por isso, removemos as duplicatas **antes** de dividir os dados.

In [9]:
# (a) removendo colunas que não devem entrar no modelo
df_modelo = df.drop(columns=COLUNAS_PARA_IGNORAR)

# Feature extra: o detector encontrou um rosto?
df_modelo["rosto_detectado"] = df_modelo["x_face"].notna().astype(int)

# (b) removendo colunas constantes (só um valor distinto)
constantes = [c for c in df_modelo.columns
              if c != COLUNA_ALVO and df_modelo[c].nunique(dropna=False) <= 1]
print("Colunas constantes removidas:", constantes)
df_modelo = df_modelo.drop(columns=constantes)

# (c) removendo linhas duplicadas
if REMOVER_DUPLICADOS:
    antes = len(df_modelo)
    df_modelo = df_modelo.drop_duplicates().reset_index(drop=True)
    print(f"Linhas antes: {antes} | depois de remover duplicadas: {len(df_modelo)}")

print()
print(df_modelo[COLUNA_ALVO].value_counts())

Colunas constantes removidas: ['documento', 'nacional', 'estudante', 'UNE']
Linhas antes: 1000 | depois de remover duplicadas: 337

label
1    171
0    166
Name: count, dtype: int64


## 5. Separando em treino e teste

- **Treino:** os dados que o modelo usa para aprender.
- **Teste:** dados que o modelo nunca viu, usados só para medir o desempenho real.

Usamos `stratify` para que a proporção entre as classes seja parecida nos dois conjuntos.

In [10]:
X = df_modelo.drop(columns=[COLUNA_ALVO])
y = df_modelo[COLUNA_ALVO]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=TAMANHO_TESTE,
    stratify=y,
    random_state=SEMENTE,
)

print("Treino:", X_treino.shape, "| Teste:", X_teste.shape)
print("Features usadas:", list(X.columns))

Treino: (269, 15) | Teste: (68, 15)
Features usadas: ['x_face', 'y_face', 'w_face', 'h_face', 'quantidade_palavras', 'estudantil', 'validade', 'unicamp', 'universidade', 'estadual', 'matricula', 'CPF', 'RG', 'data', 'rosto_detectado']


## 6. Treinando o XGBoost

O XGBoost constrói várias árvores de decisão pequenas, uma após a outra. Cada nova árvore tenta corrigir os erros das anteriores. Principais parâmetros:

- `n_estimators`: quantidade de árvores;
- `learning_rate`: o quanto cada árvore contribui (valores menores são mais cuidadosos);
- `max_depth`: profundidade máxima de cada árvore (valores menores evitam *overfitting*);
- `subsample` e `colsample_bytree`: fração de linhas e colunas sorteadas por árvore, o que também ajuda contra overfitting.

In [ ]:
modelo = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=SEMENTE,
)

modelo.fit(X_treino, y_treino)
print("Modelo treinado!")

### Validação cruzada (uma estimativa mais confiável)

Com poucos dados, o resultado de uma única divisão treino/teste pode variar bastante por sorte. A **validação cruzada** repete o treino 5 vezes, cada vez usando uma parte diferente como teste, e mostra a média e a variação.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
scores_cv = cross_val_score(modelo, X, y, cv=cv, scoring="f1")

print("F1 em cada fold:", scores_cv.round(3))
print(f"F1 médio: {scores_cv.mean():.3f} (+/- {scores_cv.std():.3f})")

## 7. Métricas no conjunto de teste

Considerando a classe `1` como a classe "positiva":

- **Acurácia:** de todas as previsões, quantas estavam certas.
- **Precisão:** quando o modelo diz "1", quantas vezes ele acerta.
- **Recall (sensibilidade):** de todos os casos que realmente são "1", quantos o modelo encontrou.
- **F1:** equilíbrio entre precisão e recall.
- **ROC AUC:** capacidade de separar as duas classes (1.0 é perfeito, 0.5 é aleatório).

In [ ]:
y_pred = modelo.predict(X_teste)
y_prob = modelo.predict_proba(X_teste)[:, 1]  # probabilidade da classe 1

metricas = {
    "Acurácia":  accuracy_score(y_teste, y_pred),
    "Precisão":  precision_score(y_teste, y_pred),
    "Recall":    recall_score(y_teste, y_pred),
    "F1":        f1_score(y_teste, y_pred),
    "ROC AUC":   roc_auc_score(y_teste, y_prob),
}

tabela_metricas = pd.DataFrame(metricas, index=["Teste"]).T.round(4)
tabela_metricas

In [ ]:
# Relatório completo por classe
print(classification_report(y_teste, y_pred, digits=3))

## 8. Matriz de confusão

A matriz mostra, em cada célula, quantos exemplos caíram em cada combinação:

- **Linhas:** a classe real.
- **Colunas:** a classe prevista pelo modelo.
- A **diagonal principal** (de cima para baixo, da esquerda para a direita) são os acertos. Fora dela estão os erros.

Em um problema binário, as quatro células são: verdadeiro negativo (VN), falso positivo (FP), falso negativo (FN) e verdadeiro positivo (VP).

In [ ]:
cm = confusion_matrix(y_teste, y_pred)
print(pd.DataFrame(cm,
                   index=["Real 0", "Real 1"],
                   columns=["Previsto 0", "Previsto 1"]))

vn, fp, fn, vp = cm.ravel()
print(f"\nVN={vn} | FP={fp} | FN={fn} | VP={vp}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Contagem absoluta
ConfusionMatrixDisplay(cm, display_labels=[0, 1]).plot(
    ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Matriz de confusão (contagem)")

# Normalizada por linha: mostra o percentual de acerto de cada classe real
ConfusionMatrixDisplay.from_predictions(
    y_teste, y_pred, normalize="true", ax=axes[1],
    cmap="Blues", colorbar=False, values_format=".0%")
axes[1].set_title("Matriz de confusão (normalizada por linha)")

plt.tight_layout()
plt.savefig("matriz_confusao.png", dpi=150)
plt.show()

## 9. Quais features mais importam?

O XGBoost informa o quanto cada feature contribuiu para as decisões das árvores. Isso ajuda a entender o que o modelo está usando.

In [ ]:
importancias = (
    pd.Series(modelo.feature_importances_, index=X.columns)
    .sort_values()
)

plt.figure(figsize=(7, 5))
importancias.plot(kind="barh", color="#4C72B0")
plt.title("Importância das features")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()

## 10. Salvando o modelo em `.joblib`

Salvamos também a **lista de colunas** usada no treino. Na hora de fazer previsões com dados novos, as colunas precisam estar exatamente nessa ordem.

In [ ]:
artefato = {
    "modelo": modelo,
    "colunas": list(X.columns),
}
joblib.dump(artefato, CAMINHO_MODELO)
print(f"Modelo salvo em: {CAMINHO_MODELO}")

## 11. Testando o carregamento do modelo salvo

Sempre vale conferir que o arquivo salvo funciona e reproduz as mesmas previsões.

In [ ]:
carregado = joblib.load(CAMINHO_MODELO)
modelo_carregado = carregado["modelo"]
colunas = carregado["colunas"]

y_pred_carregado = modelo_carregado.predict(X_teste[colunas])
print("Previsões idênticas às do modelo original:",
      bool((y_pred_carregado == y_pred).all()))

### Como usar o modelo em dados novos

```python
import joblib
import pandas as pd

artefato = joblib.load("modelo_xgboost.joblib")
modelo, colunas = artefato["modelo"], artefato["colunas"]

novos = pd.read_csv("novas_features.csv")
novos["rosto_detectado"] = novos["x_face"].notna().astype(int)

previsoes = modelo.predict(novos[colunas])
```

**Observação:** os resultados deste notebook vêm de poucos dados únicos. Por isso, vale olhar a validação cruzada (seção 6) junto com as métricas do teste, e validar o modelo com imagens realmente novas antes de colocá-lo em uso.